In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS ecommerce_catalog.silver.orders
          AS SELECT * FROM ecommerce_catalog.bronze.orders_raw
          """)

cdc_df = spark.table("ecommerce_catalog.bronze.orders_incr")

In [0]:
w = Window.partitionBy("order_id").orderBy(F.col("created_at").desc())

cdc_latest = (
    cdc_df.withColumn("rn", F.row_number().over(w))\
        .filter("rn = 1")\
            .drop("rn")
            .select(
                "order_id", "customer_id", "product_id", "order_date",
                "quantity", "unit_price", "discount_applied", "total_amount",
                "payment_mode", "coupon_code", "order_status", "delivery_city",
                "delivery_pincode", "customer_name", "customer_email", "customer_phone",
                "customer_rating_given", "return_reason", "created_at"
            )
)

silver_tbl = DeltaTable.forName(spark, "ecommerce_catalog.silver.orders")

(
    silver_tbl.alias("t")
                .merge(cdc_latest.alias("s"), "t.order_id = s.order_id")\
                    .whenMatchedUpdate(set={
                        "customer_id": "s.customer_id",
                        "product_id": "s.product_id",
                        "order_date": "s.order_date",
                        "quantity": "s.quantity",
                        "unit_price": "s.unit_price",
                        "discount_applied": "s.discount_applied",
                        "total_amount": "s.total_amount",
                        "payment_mode": "s.payment_mode",
                        "coupon_code": "s.coupon_code",
                        "order_status": "s.order_status",
                        "delivery_city": "s.delivery_city",
                        "delivery_pincode": "s.delivery_pincode",
                        "customer_name": "s.customer_name",
                        "customer_email": "s.customer_email",
                        "customer_phone": "s.customer_phone",
                        "customer_rating_given": "s.customer_rating_given",
                        "return_reason": "s.return_reason",
                        "created_at": "s.created_at"
                    })\
                        .whenNotMatchedInsert(values={
                            "order_id": "s.order_id",
                            "customer_id": "s.customer_id",
                            "product_id": "s.product_id",
                            "order_date": "s.order_date",
                            "quantity": "s.quantity",
                            "unit_price": "s.unit_price",
                            "discount_applied": "s.discount_applied",
                            "total_amount": "s.total_amount",
                            "payment_mode": "s.payment_mode",
                            "coupon_code": "s.coupon_code",
                            "order_status": "s.order_status",
                            "delivery_city": "s.delivery_city",
                            "delivery_pincode": "s.delivery_pincode",
                            "customer_name": "s.customer_name",
                            "customer_email": "s.customer_email",
                            "customer_phone": "s.customer_phone",
                            "customer_rating_given": "s.customer_rating_given",
                            "return_reason": "s.return_reason",
                            "created_at": "s.created_at",
                            "file_name": "NULL",
                            "file_size": "NULL"
                        })
                        .execute()
)

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS ecommerce_catalog.silver.products
          AS SELECT * FROM ecommerce_catalog.bronze.products_raw""")

txn_cdc = spark.table("ecommerce_catalog.bronze.products_incr").select(
    "product_id", "product_name", "brand", "category", "sub_category",
    "mrp", "discount_percent", "selling_price", "stock_quantity",
    "rating", "rating_count", "seller_name", "warehouse_city",
    "is_cruelty_free", "is_organic", "weight_grams", "launch_date",
    "status", "source_system", "created_at", "updated_at"
)
txn_tbl = DeltaTable.forName(spark, "ecommerce_catalog.silver.products")

(
    txn_tbl.alias("t")
            .merge(txn_cdc.alias("s"), "t.product_id = s.product_id")\
                .whenNotMatchedInsert(values={
                    "product_id": "s.product_id",
                    "product_name": "s.product_name",
                    "brand": "s.brand",
                    "category": "s.category",
                    "sub_category": "s.sub_category",
                    "mrp": "s.mrp",
                    "discount_percent": "s.discount_percent",
                    "selling_price": "s.selling_price",
                    "stock_quantity": "s.stock_quantity",
                    "rating": "s.rating",
                    "rating_count": "s.rating_count",
                    "seller_name": "s.seller_name",
                    "warehouse_city": "s.warehouse_city",
                    "is_cruelty_free": "s.is_cruelty_free",
                    "is_organic": "s.is_organic",
                    "weight_grams": "s.weight_grams",
                    "launch_date": "s.launch_date",
                    "status": "s.status",
                    "source_system": "s.source_system",
                    "created_at": "s.created_at",
                    "updated_at": "s.updated_at",
                    "file_name": "NULL",
                    "file_size": "NULL"
                })\
                    .execute()
                    )